# Banking System — Demos (Days 1–6)

This notebook duplicates the demonstrations from `src/main.py` and `src/simulation.py`, one section per day. Run the cells top to bottom.

> Run Jupyter from the repository root so `from src.domain import ...` resolves.

In [1]:
from datetime import datetime
from decimal import Decimal

from src.domain import (
    Bank, Client, Currency, AccountStatus, AssetClass,
    BankAccount, SavingsAccount, PremiumAccount, InvestmentAccount,
    Transaction, TransactionType, TransactionPriority,
    TransactionQueue, TransactionProcessor,
    RiskAnalyzer, AuditReporter, DomainError, UnderageError,
)

## Day 1 — Base bank account model
Abstract account + `BankAccount`: validation, statuses (active/frozen/closed), multi-currency, custom errors.

In [2]:
active = BankAccount("Alice", balance=1000, currency=Currency.USD)
frozen = BankAccount("Bob", balance=500, status=AccountStatus.FROZEN, currency=Currency.EUR)
print(active)
print(frozen)

# Operations on a frozen account are blocked.
for label, op in (("deposit", lambda: frozen.deposit(100)),
                  ("withdraw", lambda: frozen.withdraw(100))):
    try:
        op()
    except DomainError as exc:
        print(f"{label} blocked -> {type(exc).__name__}: {exc}")

# Valid operations on the active account.
active.deposit(250)
active.withdraw(400)
print(active)
print(active.get_account_info())

BankAccount | Alice | ****D604 | ACTIVE | 1000.00 USD
BankAccount | Bob | ****E606 | FROZEN | 500.00 EUR
deposit blocked -> AccountFrozenError: Account 7C7FE606 is frozen and cannot transact
withdraw blocked -> AccountFrozenError: Account 7C7FE606 is frozen and cannot transact
BankAccount | Alice | ****D604 | ACTIVE | 850.00 USD
{'account_id': '617AD604', 'owner': 'Alice', 'status': 'active', 'balance': Decimal('850.00'), 'currency': 'USD'}


## Day 2 — Advanced account types
`SavingsAccount`, `PremiumAccount`, `InvestmentAccount` with polymorphic `withdraw` / `get_account_info` / `__str__`.

In [3]:
savings = SavingsAccount("Carol", balance=1000, min_balance=200,
                         monthly_rate="0.05", currency=Currency.RUB)
print(savings)
print("interest:", savings.apply_monthly_interest(), "-> balance", savings.balance)
try:
    savings.withdraw(950)
except DomainError as exc:
    print("withdraw(950) blocked ->", type(exc).__name__, exc)
print("withdraw(300) ok -> balance", savings.withdraw(300))

premium = PremiumAccount("Dave", balance=100, overdraft_limit=500,
                         transaction_fee=5, currency=Currency.USD)
print(premium)
print("withdraw(400) with overdraft -> balance", premium.withdraw(400))

invest = InvestmentAccount("Eve", balance=10000, currency=Currency.EUR)
invest.invest(AssetClass.STOCKS, 4000)
invest.invest(AssetClass.BONDS, 3000)
invest.invest(AssetClass.ETF, 2000)
print(invest)
print("projected yearly growth:", invest.project_yearly_growth(), invest.currency.value)

SavingsAccount | Carol | ****D448 | ACTIVE | 1000.00 RUB | min=200.00, rate=0.05
interest: 50.00 -> balance 1050.00
withdraw(950) blocked -> InsufficientFundsError Withdrawal would breach the minimum balance of 200.00; available above minimum is 850.00
withdraw(300) ok -> balance 750.00
PremiumAccount | Dave | ****759F | ACTIVE | 100.00 USD | overdraft=500.00, fee=5.00
withdraw(400) with overdraft -> balance -305.00
InvestmentAccount | Eve | ****CB7D | ACTIVE | 1000.00 EUR | portfolio=9000.00 EUR
projected yearly growth: 660.00 EUR


## Day 3 — Bank system & client management
`Client` (age check, PIN) and `Bank` (accounts, authentication, security, analytics).

In [4]:
bank = Bank("Demo Bank")

try:
    Client("Young Tom", 16, pin="0000")
except UnderageError as exc:
    print("underage rejected ->", exc)

alice = bank.add_client(Client("Alice Smith", 30, pin="1234", client_id="ALICE001"))
bob = bank.add_client(Client("Bob Jones", 45, pin="4321", client_id="BOB002"))
a1 = bank.open_account(alice.client_id, "savings", balance=5000, min_balance=500,
                       monthly_rate="0.04", currency=Currency.USD)
bank.open_account(alice.client_id, "premium", balance=2000, overdraft_limit=1000,
                  transaction_fee=10, currency=Currency.USD)
bank.open_account(bob.client_id, "investment", balance=8000, currency=Currency.EUR)
print(alice, "| accounts:", alice.account_numbers)

# Three wrong PINs block the client.
for _ in range(3):
    bank.authenticate_client("BOB002", "0000")
print("Bob blocked:", bob.is_blocked, "| flags:", bob.suspicious_flags)

print("Total balance per currency:", bank.get_total_balance())
print("Clients ranking:", bank.get_clients_ranking())

underage rejected -> Client must be at least 18; got 16
Client ALICE001 | Alice Smith | ACTIVE | 2 account(s) | accounts: ['815AD054', '392A489E']
Bob blocked: True | flags: ['failed login attempt #1', 'failed login attempt #2', 'failed login attempt #3', 'blocked: too many failed login attempts']
Total balance per currency: {'USD': Decimal('7000.00'), 'EUR': Decimal('8000.00')}
Clients ranking: [{'client_id': 'BOB002', 'full_name': 'Bob Jones', 'total': Decimal('8000.00')}, {'client_id': 'ALICE001', 'full_name': 'Alice Smith', 'total': Decimal('7000.00')}]


## Day 4 — Transaction system
`Transaction`, `TransactionQueue` (priority/deferred/cancel), `TransactionProcessor` (fees, conversion, retries).

In [5]:
bank = Bank("Demo Bank")
bank.add_client(Client("Alice", 30, pin="1", client_id="C1"))
bank.add_client(Client("Bob", 30, pin="2", client_id="C2"))
usd = bank.open_account("C1", "bank", balance=1000, currency=Currency.USD)
eur = bank.open_account("C2", "bank", balance=1000, currency=Currency.EUR)

processor = TransactionProcessor(bank)
queue = TransactionQueue()
transactions = [
    Transaction(TransactionType.DEPOSIT, 200, Currency.USD, receiver=usd.account_id),
    Transaction(TransactionType.WITHDRAWAL, 150, Currency.USD, sender=usd.account_id),
    Transaction(TransactionType.TRANSFER, 100, Currency.USD,
                sender=usd.account_id, receiver=eur.account_id),
    Transaction(TransactionType.EXTERNAL_TRANSFER, 100, Currency.USD,
                sender=usd.account_id, receiver="EXT-OUT"),
    Transaction(TransactionType.WITHDRAWAL, 999999, Currency.USD, sender=usd.account_id),
    Transaction(TransactionType.DEPOSIT, 25, Currency.USD, receiver=usd.account_id,
                priority=TransactionPriority.HIGH),
    Transaction(TransactionType.DEPOSIT, 5, Currency.USD, receiver=usd.account_id,
                transaction_id="CANCELLED-1"),
]
for tx in transactions:
    queue.add(tx)
queue.cancel("CANCELLED-1")

for tx in processor.process_queue(queue):
    suffix = f"  ({tx.failure_reason})" if tx.failure_reason else ""
    print(tx, suffix)
print("Final balances: USD", usd.balance, "EUR", eur.balance)

transaction 7D820EDA failed on attempt 1: InsufficientFundsError: Cannot withdraw 999999.00; balance is 874.00


Tx 1E31E0B2 | deposit | 25.00 USD | COMPLETED 
Tx D91813D5 | deposit | 200.00 USD | COMPLETED 
Tx 499B6FDC | withdrawal | 150.00 USD | COMPLETED 
Tx 3E8E5B82 | transfer | 100.00 USD | COMPLETED 
Tx 84ECF648 | external_transfer | 100.00 USD | COMPLETED 
Tx 7D820EDA | withdrawal | 999999.00 USD | FAILED   (Cannot withdraw 999999.00; balance is 874.00)
Final balances: USD 874.00 EUR 1092.59


## Day 5 — Audit & risk analysis
`RiskAnalyzer` flags suspicious operations; the processor blocks high-risk ones; `AuditReporter` builds reports.

In [6]:
midday = datetime(2026, 6, 9, 12, 0, 0)
night = datetime(2026, 6, 9, 2, 0, 0)
bank = Bank("Demo Bank", now=lambda: midday)
bank.add_client(Client("Alice", 30, pin="1", client_id="C1"))
account = bank.open_account("C1", "bank", balance=100000, currency=Currency.USD)

analyzer = RiskAnalyzer(large_amount=10000, frequency_limit=3)
processor = TransactionProcessor(bank, risk=analyzer, now=lambda: night)
ops = [
    Transaction(TransactionType.WITHDRAWAL, 100, Currency.USD, sender=account.account_id),
    Transaction(TransactionType.WITHDRAWAL, 200, Currency.USD, sender=account.account_id),
    Transaction(TransactionType.EXTERNAL_TRANSFER, 50000, Currency.USD,
                sender=account.account_id, receiver="EXT-NEW"),
    Transaction(TransactionType.WITHDRAWAL, 50, Currency.USD, sender=account.account_id),
    Transaction(TransactionType.WITHDRAWAL, 60, Currency.USD, sender=account.account_id),
]
for tx in ops:
    processor.process(tx)
    print(tx, ("-> " + tx.failure_reason) if tx.failure_reason else "")

reporter = AuditReporter(processor.audit)
print("\nSuspicious operations:", len(reporter.suspicious_operations()))
print("Client risk profile:", reporter.client_risk_profile(bank, "C1"))
print("Error statistics:", reporter.error_statistics())

Tx 8F875FDA | withdrawal | 100.00 USD | COMPLETED 
Tx C8D97837 | withdrawal | 200.00 USD | COMPLETED 
Tx 38DA88E9 | external_transfer | 50000.00 USD | FAILED -> blocked by risk analysis: large amount (50000.00), night operation (02h), frequent operations, transfer to new account (EXT-NEW)
Tx 4C02BC48 | withdrawal | 50.00 USD | FAILED -> blocked by risk analysis: night operation (02h), frequent operations
Tx 7320EF14 | withdrawal | 60.00 USD | FAILED -> blocked by risk analysis: night operation (02h), frequent operations

Suspicious operations: 8
Client risk profile: {'client_id': 'C1', 'full_name': 'Alice', 'assessed_operations': 5, 'counts_by_level': {'LOW': 0, 'MEDIUM': 2, 'HIGH': 3}, 'highest_level': 'HIGH', 'reasons': ['frequent operations', 'large amount (50000.00)', 'night operation (02h)', 'transfer to new account (EXT-NEW)']}
Error statistics: {'total_errors': 3, 'by_type': {'RiskBlocked': 3}}


## Day 6 — Comprehensive simulation
The full capstone: a bank with 7 clients and 12 accounts, ~40 transactions (some erroneous, some suspicious), user scenarios, and reports.

In [7]:
from src.simulation import run
run()

transaction 3A8B4C9C failed on attempt 1: InsufficientFundsError: Cannot withdraw 10000000.00; balance is 10063.31


transaction C7D25285 failed on attempt 1: AccountFrozenError: Account 6CE06738 is frozen and cannot transact


transaction 4CAFBA29 failed on attempt 1: AccountNotFoundError: No account with id 'GHOST-404'


=== Initialization ===
Bank: Mega Bank | clients: 7 | accounts: 12

=== Simulation: 40 transactions queued ===

  [OK]  Tx 6B6A2975 | deposit | 110.00 EUR | COMPLETED
  [OK]  Tx DBFCA356 | transfer | 372.00 EUR | COMPLETED
  [OK]  Tx E3BD3D22 | withdrawal | 345.00 USD | COMPLETED
  [OK]  Tx 983C1978 | withdrawal | 163.00 USD | COMPLETED
  [OK]  Tx 2162C2EC | deposit | 346.00 RUB | COMPLETED
  [OK]  Tx 949D1585 | deposit | 170.00 KZT | COMPLETED
  [OK]  Tx 2D3E8D07 | withdrawal | 467.00 CNY | COMPLETED
  [OK]  Tx 41403387 | withdrawal | 347.00 EUR | COMPLETED
  [OK]  Tx 79DC5022 | withdrawal | 240.00 CNY | COMPLETED
  [OK]  Tx D45B938A | deposit | 114.00 EUR | COMPLETED
  [OK]  Tx 97CA7B92 | withdrawal | 155.00 USD | COMPLETED
  [OK]  Tx 553961F0 | deposit | 487.00 KZT | COMPLETED
  [OK]  Tx 4DB74011 | withdrawal | 349.00 KZT | COMPLETED
  [OK]  Tx 57082BB1 | transfer | 177.00 EUR | COMPLETED
  [OK]  Tx 10CE1898 | withdrawal | 91.00 CNY | COMPLETED
  [OK]  Tx D7D1DDB3 | withdrawal | 303